# PayShield AI — Model Comparison

## AI-Powered Payment Success Optimization

### Objective

Compare different machine learning models for predicting
upcoming payment failure risk.

Models:

1. Logistic Regression
2. Random Forest
3. Gradient Boosting

Evaluation metrics:

- Precision
- Recall
- F1-score
- ROC-AUC
- PR-AUC

Because the risk class is highly imbalanced, accuracy will
not be used as the primary metric.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

In [2]:
df = pd.read_csv(
    "../data/processed/ml_dataset.csv"
)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (50944, 22)


,transaction_count,failure_rate,timeout_rate,avg_latency,max_latency,p95_latency,bank_error_rate,avg_amount,max_amount,hour,...,previous_failure_rate,previous_latency,previous_timeout_rate,failure_rate_change,latency_change,timeout_rate_change,rolling_failure_rate,rolling_latency,rolling_timeout_rate,risk_target
0,2,0.0,0.0,881.935,883.95,883.7485,0.0,244.72,320.59,0,...,0.0,1447.700,0.0,0.0,-565.765,0.0,0.0,1164.817500,0.0,0
1,1,0.0,0.0,882.880,882.88,882.8800,0.0,321.00,321.00,1,...,0.0,881.935,0.0,0.0,0.945,0.0,0.0,1070.838333,0.0,0
2,1,0.0,0.0,751.670,751.67,751.6700,0.0,395.70,395.70,1,...,0.0,882.880,0.0,0.0,-131.210,0.0,0.0,838.828333,0.0,0
3,1,0.0,0.0,1161.920,1161.92,1161.9200,0.0,32.14,32.14,1,...,0.0,751.670,0.0,0.0,410.250,0.0,0.0,932.156667,0.0,0
4,1,0.0,0.0,709.700,709.70,709.7000,0.0,920.47,920.47,2,...,0.0,1161.920,0.0,0.0,-452.220,0.0,0.0,874.430000,0.0,0


In [3]:
X = df.drop(columns=["risk_target"])
y = df["risk_target"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (50944, 21)
Target shape: (50944,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 40755
Testing samples: 10189


In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

lr_model.fit(
    X_train_scaled,
    y_train
)

lr_pred = lr_model.predict(X_test_scaled)

lr_prob = lr_model.predict_proba(
    X_test_scaled
)[:, 1]

In [7]:
lr_roc = roc_auc_score(
    y_test,
    lr_prob
)

lr_pr = average_precision_score(
    y_test,
    lr_prob
)

print("Logistic Regression")
print("-------------------")
print("ROC-AUC:", lr_roc)
print("PR-AUC :", lr_pr)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        lr_pred,
        target_names=["No Risk", "Risk"]
    )
)

Logistic Regression
-------------------
ROC-AUC: 0.9492182543598708
PR-AUC : 0.8333366504377651

Classification Report:
              precision    recall  f1-score   support

     No Risk       1.00      0.98      0.99     10133
        Risk       0.19      0.88      0.31        56

    accuracy                           0.98     10189
   macro avg       0.59      0.93      0.65     10189
weighted avg       0.99      0.98      0.99     10189



In [8]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

rf_pred = rf_model.predict(X_test)

rf_prob = rf_model.predict_proba(
    X_test
)[:, 1]

In [9]:
rf_roc = roc_auc_score(
    y_test,
    rf_prob
)

rf_pr = average_precision_score(
    y_test,
    rf_prob
)

print("Random Forest")
print("-------------")
print("ROC-AUC:", rf_roc)
print("PR-AUC :", rf_pr)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        rf_pred,
        target_names=["No Risk", "Risk"]
    )
)

Random Forest
-------------
ROC-AUC: 0.9334890597905006
PR-AUC : 0.8003917898934033

Classification Report:
              precision    recall  f1-score   support

     No Risk       1.00      1.00      1.00     10133
        Risk       0.95      0.73      0.83        56

    accuracy                           1.00     10189
   macro avg       0.98      0.87      0.91     10189
weighted avg       1.00      1.00      1.00     10189



In [10]:
gb_model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model.fit(
    X_train,
    y_train
)

gb_pred = gb_model.predict(X_test)

gb_prob = gb_model.predict_proba(
    X_test
)[:, 1]

In [15]:
gb_roc = roc_auc_score(
    y_test,
    gb_prob
)

gb_pr = average_precision_score(
    y_test,
    gb_prob
)

print("Gradient Boosting")
print("-----------------")
print("ROC-AUC:", gb_roc)
print("PR-AUC :", gb_pr)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        gb_pred,
        target_names=["No Risk", "Risk"]
    )
)

Gradient Boosting
-----------------
ROC-AUC: 0.9569608140305368
PR-AUC : 0.5526138427725349

Classification Report:
              precision    recall  f1-score   support

     No Risk       1.00      1.00      1.00     10133
        Risk       0.74      0.75      0.74        56

    accuracy                           1.00     10189
   macro avg       0.87      0.87      0.87     10189
weighted avg       1.00      1.00      1.00     10189



In [16]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "ROC_AUC": [
        lr_roc,
        rf_roc,
        gb_roc
    ],
    "PR_AUC": [
        lr_pr,
        rf_pr,
        gb_pr
    ]
})

comparison = comparison.sort_values(
    "PR_AUC",
    ascending=False
)

comparison

,Model,ROC_AUC,PR_AUC
0,Logistic Regression,0.949218,0.833337
1,Random Forest,0.933489,0.800392
2,Gradient Boosting,0.956961,0.552614


In [17]:
models = {
    "Logistic Regression": (lr_pred, lr_prob),
    "Random Forest": (rf_pred, rf_prob),
    "Gradient Boosting": (gb_pred, gb_prob)
}

results = []

for name, (pred, prob) in models.items():

    report = classification_report(
        y_test,
        pred,
        output_dict=True
    )

    results.append({
        "Model": name,
        "Precision": report["1"]["precision"],
        "Recall": report["1"]["recall"],
        "F1": report["1"]["f1-score"],
        "ROC-AUC": roc_auc_score(y_test, prob),
        "PR-AUC": average_precision_score(y_test, prob)
    })

results_df = pd.DataFrame(results)

results_df.sort_values(
    "PR-AUC",
    ascending=False
)

,Model,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Logistic Regression,0.186312,0.875000,0.307210,0.949218,0.833337
1,Random Forest,0.953488,0.732143,0.828283,0.933489,0.800392
2,Gradient Boosting,0.736842,0.750000,0.743363,0.956961,0.552614


In [18]:
rf_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_
})

rf_importance = rf_importance.sort_values(
    "importance",
    ascending=False
)

rf_importance.head(15)

,feature,importance
19,rolling_latency,0.250088
3,avg_latency,0.174527
13,previous_latency,0.161687
5,p95_latency,0.112608
4,max_latency,0.111822
18,rolling_failure_rate,0.044404
16,latency_change,0.021587
9,hour,0.019586
20,rolling_timeout_rate,0.019320
12,previous_failure_rate,0.014693


In [19]:
print("Top Random Forest features:")
print(
    rf_importance.head(10).to_string(
        index=False
    )
)

Top Random Forest features:
              feature  importance
      rolling_latency    0.250088
          avg_latency    0.174527
     previous_latency    0.161687
          p95_latency    0.112608
          max_latency    0.111822
 rolling_failure_rate    0.044404
       latency_change    0.021587
                 hour    0.019586
 rolling_timeout_rate    0.019320
previous_failure_rate    0.014693


In [20]:
best_model_name = results_df.loc[
    results_df["PR-AUC"].idxmax(),
    "Model"
]

print("Best model based on PR-AUC:")
print(best_model_name)

Best model based on PR-AUC:
Logistic Regression


In [21]:
if best_model_name == "Logistic Regression":
    best_pred = lr_pred

elif best_model_name == "Random Forest":
    best_pred = rf_pred

else:
    best_pred = gb_pred

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        best_pred
    )
)


Confusion Matrix:
[[9919  214]
 [   7   49]]
